# Pipeline completo: coleta, limpeza, análise e exportação

Este notebook percorre, uma etapa por vez, o pipeline do script `examples/04_pipeline_completo.py`. O pipeline coleta no IPEA as publicações sobre dois temas de saúde, limpa os dados, faz uma análise descritiva e exporta as tabelas. As funções vêm do script; aqui o foco é ver o que cada etapa recebe, o que devolve e por que ela existe.

**Fonte:** [IPEA, central de conteúdo](https://www.ipea.gov.br/portal/coluna-5/central-de-conteudo/busca-publicacoes)

**Para rodar:** o notebook importa funções de `examples/`, então precisa de um clone do repositório e deve ser aberto de dentro da pasta `notebooks/`. No Google Colab, rode antes, numa célula: `!pip install git+https://github.com/bdcdo/raspe.git`, `!git clone https://github.com/bdcdo/raspe.git` e `%cd raspe/notebooks`.

## Preparação

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# A pasta examples/ fica ao lado de notebooks/. Incluí-la no caminho de importação deixa o
# notebook usar as funções do script sem copiá-las.
sys.path.insert(0, str(Path.cwd().parent / "examples"))

# O nome do script começa com número, o que o comando import não aceita; importlib aceita.
pipeline = importlib.import_module("04_pipeline_completo")
pipeline.TERMOS

['judicialização da saúde', 'saúde suplementar']

## 1. Coleta

`pipeline.coletar()` busca cada termo separadamente com `coletar_termo()`. Se o site falhar com erro de rede (tempo esgotado, conexão recusada, erro HTTP), a função espera e tenta de novo, até `pipeline.TENTATIVAS` vezes, dobrando a espera a cada falha. Se um termo não voltar, os outros seguem. Para cada termo, a função anota uma linha no registro da coleta.

Para o notebook rodar rápido, a coleta fica nas duas primeiras páginas de cada termo, até 20 publicações por termo. O script coleta até 7 páginas.

In [2]:
bruto, registro = pipeline.coletar(pipeline.TERMOS, paginas=range(1, 3))
registro

Coletando 'judicialização da saúde'


Baixando documentos:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando documentos: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]

Baixando documentos: 100%|██████████| 1/1 [00:02<00:00,  2.26s/it]

Processando documentos:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processando documentos:  50%|█████     | 1/2 [00:00<00:00,  1.04it/s]

Processando documentos: 100%|██████████| 2/2 [00:01<00:00,  1.27it/s]

Processando documentos: 100%|██████████| 2/2 [00:01<00:00,  1.23it/s]

  ok: 20 registros


,termo,situacao,registros,tentativas,erro,coletado_em,versao_raspe
0,judicialização da saúde,ok,4,1,,2026-10-21T22:24:25,0.1.0
1,saúde suplementar,ok,20,1,,2026-10-21T22:24:30,0.1.0


O registro guarda quando a coleta rodou, com que versão do raspe, quantos registros vieram e se houve erro. Ele vai para o disco junto com os dados: sem ele, uma tabela menor do que o esperado não diz se a busca tinha menos resultados ou se a coleta falhou no meio.

In [3]:
bruto.head()

,titulo,link,autores,data,assuntos,termo_busca
0,Pesquisa assistência farmacêutica no SUS : gas...,https://www.ipea.gov.br/portal/publicacao-item...,,Mai-2025,Temas:\nAdministração Pública. Governo. Estado...,judicialização da saúde
1,"Direito à saúde no Brasil : seus contornos, ju...",https://www.ipea.gov.br/portal/publicacao-item...,,Mar-2020,"Temas:\nDireito. Legislação, \nSaúde, \n ...",judicialização da saúde
2,Dimensionamento de força de trabalho : a exper...,https://www.ipea.gov.br/portal/publicacao-item...,,2019,Temas:\nAdministração Pública. Governo. Estado...,judicialização da saúde
3,Evolução do gasto com medicamentos do Sistema ...,https://www.ipea.gov.br/portal/publicacao-item...,,Jan-2018,"Temas:\nSaúde, \n ...",judicialização da saúde
4,Inflação dos planos de saúde - 2000-2018,https://www.ipea.gov.br/portal/publicacao-item...,,Mai-2019,"Temas:\nSaúde, \n ...",saúde suplementar


### Por que limpar

Os dados brutos chegam como o site os publica. Duas colunas mostram o problema: a data vem em formatos diferentes na mesma coluna, e os temas vêm num texto só, com quebras de linha e repetições.

In [4]:
# Troca cada dígito por 9 para mostrar só o formato de cada data.
bruto["data"].str.replace(r"\d", "9", regex=True).value_counts()

data
9999        11
Mai-9999     2
Mar-9999     2
Jan-9999     2
Ago-9999     2
Set-9999     2
Dez-9999     1
Out-9999     1
-9999        1
Name: count, dtype: int64

In [5]:
print(repr(bruto.loc[0, "assuntos"]))

'Temas:\nAdministração Pública. Governo. Estado, \nSaúde, \n                                                                    \n                                Economia da Saúde, \nAdministração Pública. Governo. Estado, \n                                                                    \n                                Economia da Saúde, \nSaúde,'


## 2. Limpeza

`pipeline.limpar()` faz quatro coisas:

1. normaliza os espaços do título e dos autores;
2. converte a data em ano e mês com `converter_data()`;
3. transforma os temas numa lista sem repetição com `separar_temas()`;
4. junta numa linha só a publicação que mais de um termo encontrou, e guarda na coluna `termos` todos os termos que a acharam.

Primeiro, as duas funções auxiliares aplicadas a valores reais da coleta:

In [6]:
for data in bruto["data"].drop_duplicates().head(6):
    print(f"{data!r:12} -> (ano, mês) = {pipeline.converter_data(data)}")
print()
print(pipeline.separar_temas(bruto.loc[0, "assuntos"]))

'Mai-2025'   -> (ano, mês) = (2025, 5)
'Mar-2020'   -> (ano, mês) = (2020, 3)
'2019'       -> (ano, mês) = (2019, None)
'Jan-2018'   -> (ano, mês) = (2018, 1)
'Mai-2019'   -> (ano, mês) = (2019, 5)
'2018'       -> (ano, mês) = (2018, None)

Administração Pública. Governo. Estado; Saúde; Economia da Saúde


In [7]:
limpo = pipeline.limpar(bruto)
print(f"Linhas brutas: {len(bruto)}")
print(f"Publicações depois da limpeza: {len(limpo)}")
print(f"Publicações achadas pelos dois termos: {limpo['termos'].str.contains('; ').sum()}")
print(f"Publicações sem ano legível: {limpo['ano'].isna().sum()}")
limpo.head()

Linhas brutas: 24
Publicações depois da limpeza: 24
Publicações achadas pelos dois termos: 0
Publicações sem ano legível: 0


,titulo,link,autores,data,ano,mes,temas,termos
0,Pesquisa assistência farmacêutica no SUS : gas...,https://www.ipea.gov.br/portal/publicacao-item...,,Mai-2025,2025,5,Administração Pública. Governo. Estado; Saúde;...,judicialização da saúde
1,"Direito à saúde no Brasil : seus contornos, ju...",https://www.ipea.gov.br/portal/publicacao-item...,,Mar-2020,2020,3,Direito. Legislação; Saúde; Economia da Saúde,judicialização da saúde
2,Dimensionamento de força de trabalho : a exper...,https://www.ipea.gov.br/portal/publicacao-item...,,2019,2019,<NA>,Administração Pública. Governo. Estado; Saúde;...,judicialização da saúde
3,Evolução do gasto com medicamentos do Sistema ...,https://www.ipea.gov.br/portal/publicacao-item...,,Jan-2018,2018,1,Saúde; Economia da Saúde,judicialização da saúde
4,Inflação dos planos de saúde - 2000-2018,https://www.ipea.gov.br/portal/publicacao-item...,,Mai-2019,2019,5,Saúde; Economia da Saúde,saúde suplementar


Conferir as contagens de cada etapa é o que garante que a limpeza não perdeu linhas em silêncio. Se o número de publicações sem ano subir numa coleta futura, o IPEA passou a publicar a data num formato novo, e `converter_data()` precisa aprender esse formato.

## 3. Análise

In [8]:
tabelas = pipeline.analisar(limpo)
tabelas["por_ano"]

,ano,publicacoes
0,2015,1
1,2016,2
2,2017,1
3,2018,5
4,2019,4
5,2020,3
6,2021,3
7,2022,2
8,2023,1
9,2024,1


In [9]:
tabelas["por_termo"]

,termo,publicacoes
0,saúde suplementar,20
1,judicialização da saúde,4


Os temas vêm da classificação do próprio IPEA, e uma publicação costuma ter mais de um. Por isso a soma da tabela abaixo passa do número de publicações.

In [10]:
tabelas["temas"]

,tema,publicacoes
0,Saúde,20
1,Economia da Saúde,20
2,Administração Pública. Governo. Estado,8
3,Economia. Desenvolvimento Econômico,3
4,Direito. Legislação,1
5,Sistema Monetário. Finanças. Bancos,1
6,Gênero e Raça,1
7,Transportes,1


## 4. Exportação

`pipeline.exportar()` salva cada tabela num CSV em UTF-8 com BOM, a marca que faz o Excel abrir o arquivo com os acentos certos. Aqui os arquivos vão para a pasta `saidas/`, ao lado do notebook, para não sobrescrever os dados de exemplo que o script guarda em `examples/datasets/`.

In [11]:
arquivos = pipeline.exportar(
    {"bruto": bruto, "registro_da_coleta": registro, "publicacoes": limpo, **tabelas},
    Path("saidas"),
    prefixo="10_ipea_",
)
for arquivo in arquivos:
    print(f"{arquivo.name:32} {arquivo.stat().st_size:>7} bytes")

10_ipea_bruto.csv                  11096 bytes
10_ipea_registro_da_coleta.csv       184 bytes
10_ipea_publicacoes.csv             6637 bytes
10_ipea_por_ano.csv                   96 bytes
10_ipea_por_termo.csv                 72 bytes
10_ipea_temas.csv                    231 bytes


## O que acontece quando o site cai

Para ver o tratamento de erro sem esperar uma queda real, a célula abaixo simula uma falha de conexão. Dentro do bloco `with`, o método do raspador que faz a requisição é trocado por um que sempre levanta `ConnectionError`. A espera entre tentativas cai para 1 segundo, para a célula terminar rápido, e volta ao valor do script no fim.

In [12]:
from unittest import mock

import requests

from raspe.scrapers.ipea import ScraperIpea

falha = requests.ConnectionError("conexão recusada (simulada)")
with mock.patch.object(pipeline, "ESPERA_INICIAL", 1), mock.patch.object(ScraperIpea, "_set_r", side_effect=falha):
    dados, linha = pipeline.coletar_termo("saúde suplementar")

print(f"Linhas coletadas: {len(dados)}")
pd.Series(linha)

  Falha na tentativa 1 (ConnectionError); nova tentativa em 1 s.


  Falha na tentativa 2 (ConnectionError); nova tentativa em 2 s.


Linhas coletadas: 0


termo                                      saúde suplementar
situacao                                              falhou
registros                                                  0
tentativas                                                 3
erro            ConnectionError: conexão recusada (simulada)
coletado_em                              2026-10-21T22:24:38
versao_raspe                                           0.1.0
dtype: object

A coleta não interrompe o notebook: depois das tentativas, o termo volta vazio e o registro diz o que houve. Num pipeline com vários termos, os outros seguem, e o registro salvo mostra qual termo precisa ser coletado de novo.

## Para ir além

- O script `examples/04_pipeline_completo.py` roda o mesmo pipeline de ponta a ponta, com até 7 páginas por termo, e salva as tabelas em `examples/datasets/`.
- Para outro tema, troque `pipeline.TERMOS` por uma lista sua antes de chamar `pipeline.coletar()`.